# 📐 level_flow — multi-week levels × today's activity, long and short, flat by the close

**The idea.** Horizontal levels built from the last few weeks of swing highs and lows are where intraday trends get *interrupted* more often than chance — but a level says nothing about *which way*: take-profit orders cluster **at** it (trends reverse there, `polarity: bounce`), stop orders cluster just **beyond** it (a trend that gets through accelerates, `polarity: break`). Raw geometry has no edge net of costs; what the literature does find is that intraday price signals only work under **elevated activity**. So each session a watchlist of the most active names *with a tested level within reach* is frozen at `scan_minute`, and only those may trade — a day with no such name is a flat day, the intended default. Risk is thesis invalidation beyond the level, an R-multiple target, a time stop, a maximum stop distance (wider setups are skipped), per-entry sizing off the stop, an SSR guard on shorts and a mandatory session-close flat. Full thesis, sources and the pre-registered read in [README.md](./README.md).

**Multiple timeframes.** The execution bar is the panel's (`data.resample` derives 5m/15m bars from the stored 1m history); `tf_levels` is the candle whose swings define the levels; `signal_tf` the candle on whose closes the trigger is read; `trend_tf` the higher-timeframe EMA trend a `trend_gate` can require the trade to run with or against. Nothing from an unfinished candle ever reaches a decision. All time knobs are minutes.

**What this notebook decides, and in which order.** §5b measures the *naked* trigger — forward return from the fill, market removed, signed as positioned — by polarity, side, touch count, relative volume and trend state, before any backtest is trusted. The rule is written down before the study runs: **the polarity whose uncapped row is positive with t > 2 at 60 and 120 bars becomes the shipped value; neither, and the strategy is written up in IDEAS.md instead of optimized.** §6 then optimizes `alpha_cagr` (the return after the beta the book actually ran — a book flat most of the day cannot out-earn a fully invested +47%/yr basket, and is not meant to; `excess_cagr` is reported everywhere) against `volatility`, scored **warm** from `optuna.score_from` so the level warm-up is not charged against an invested benchmark. Everything judged here is in-sample; [`level_flow_diagnostics.ipynb`](./level_flow_diagnostics.ipynb) runs the walk-forward, the untouched holdout and the verdict.

**The universe.** `Morning200` — 200 names ranked point-in-time by dollar volume over 2025-01 → 2025-06, before the window (see `gap_and_go` for why `HighOctaneTop20`'s +460%/yr hurdle voided two studies). The 1-minute backfill list was assembled in 2026, so pool *membership* is survivorship-tinged; only the ranking within it is point-in-time.

Every knob lives in [`config.yaml`](./config.yaml); this notebook only *drives* the shared engine — **nothing here defines behavior**.

## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import backtest as bt, jobs, optimize as opt, strategies
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple import plots

strat = strategies.load("level_flow")
compose = strat.signals.__globals__["compose"]      # the intermediates signals() computes
SCORE_FROM = strat.optuna.get("score_from")          # every number below is scored WARM from here
FLOOR = strat.optuna["pick_min_trades_per_year"]     # the ONE picking floor, both notebooks
print("data :", strat.data)
print("objectives:", opt.objectives_for(strat), "· scored from", SCORE_FROM)

In [ ]:
d = strat.data
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]),
                         d["bar"], d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges,
                   resample=d.get("resample"))
print(f"panel: {panel.close.shape[0]:,} {panel.bar} bars x {len(panel.symbols)} symbols "
      f"({panel.close.index[0]:%Y-%m-%d} → {panel.close.index[-1]:%Y-%m-%d})")

## 2 · Strategy overview — hyperparameters and search ranges

| group | knobs | what they decide |
|---|---|---|
| levels | `tf_levels`, `pivot_k`, `level_days`, `bin_pct`, `min_touches` | which candle's swings, how far back, how wide a bin, how many pivots make a level (1 = any swing: the control) |
| timeframes | `signal_tf`, `trend_tf`, `trend_len`, `trend_gate` | the candle the trigger is read on; the higher-timeframe trend and whether to trade with / against it |
| trigger | `polarity`, `touch_tol`, `touch_minutes`, `confirm_minutes`, `break_tol`, `vwap_gate`, `range_used_max` | bounce or break, the touch band and reclaim, the break margin, the VWAP side, "already spent its range" |
| scan | `scan_minute`, `vol_mult`, `rank_by`, `max_candidates`, `reach_atr`, floors | when the watchlist freezes, who qualifies (0 = no activity gate: the control), how it is ordered, how many seats |
| regime | `regime`, `regime_days`, `warmup_sessions` | the market-state gate; the pinned dead zone every trial shares |
| risk | `sl_atr`, `max_stop`, `tp_ratio`, `sl_trail`, `horizon`, `risk_per_trade`, `ssr_guard`, `eod_exit` | stop beyond the level, skip-if-wider, target, trail, time stop, size off the stop, no SSR shorts, flat by the close |

Every distance is in **session-scale ATRs**; every time knob in **minutes**. The table below shows, per parameter: the **default** (§3), the **sweep grid** (§5) and the **Optuna range** (§6). `—` = the stage keeps the default.

In [ ]:
import pandas as pd

def _fmt_space(spec):
    """One readable line for an optuna.space entry."""
    if spec["type"] == "categorical":
        return " | ".join(map(str, spec["choices"]))
    tail = " (log)" if spec.get("log") else (f", step {spec['step']}" if spec.get("step") else "")
    return f"{spec['low']} – {spec['high']}{tail}"

pd.DataFrame({
    "default": pd.Series(strat.params),
    "sweep grid (§5)": pd.Series({k: ", ".join(map(str, v)) for k, v in strat.sweep.items()}),
    "optuna search range (§6)": pd.Series({k: _fmt_space(s)
                                           for k, s in strat.optuna["space"].items()}),
    "doc": pd.Series(strat.docs),
}).fillna("—")

## 3 · Single backtest with the config defaults — against the baseline

Engine rules (identical for every strategy): signals execute at the **next bar's adjusted open**, realistic costs are always charged, and every number is scored **warm from `optuna.score_from`** — signals run on the whole panel, the book and the benchmark only from that date, so the 20-session level warm-up is not a flat stretch charged against an invested benchmark.

> **Capital model:** ONE shared \$10k wallet. Each entry deploys `min(1/max_candidates, risk_per_trade / stop)` of the **free** cash (the engine's per-entry `cash_frac` frame), so one stop-out costs at most `risk_per_trade` of the book, gaps aside. Percent sizing is of the free cash: five same-day entries at a 1/5 slot deploy 20%, 16%, 12.8%, … — never 100%.

The baseline is `strategies/baseline` — equal-weight buy-and-hold of the same names (test-pinned to the engine's benchmark). `alpha_cagr` is what is left of the strategy's return after its `beta` to that basket; `excess_cagr` is the plain difference and will be negative for a mostly-flat book.

In [ ]:
import plotly.graph_objects as go

INIT_CASH = settings.backtest.init_cash
baseline = strategies.load("baseline")


def scored(params, *, strategy=strat):
    """Portfolio + the evaluate() stats, warm from SCORE_FROM (one run, one pricing)."""
    pf, window, fees, slip = bt._run_scored(strategy, panel, params, settings, SCORE_FROM)
    stats = bt.portfolio_stats(pf)
    stats |= bt.active_stats(pf, window, fees=fees, slippage=slip, fill=strategy.fill,
                             total_return=stats["total_return"])
    return pf, stats


def overlay(pf, label, title):
    """Cumulative return of a book against the baseline, both from SCORE_FROM."""
    fig = go.Figure()
    fig.add_trace(plots._scatter(
        bt.equity_curve(pf_base, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
        name="baseline (buy & hold)", line={"color": "#898781", "dash": "dash", "width": 1.5},
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>%{y:+.1%}<extra>baseline</extra>"))
    fig.add_trace(plots._scatter(
        bt.equity_curve(pf, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
        name=label, line={"color": plots.SERIES[0], "width": 1.5},
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>%{y:+.1%}<extra>" + label + "</extra>"))
    plots._base(fig, title=title, height=420)
    fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
    return fig


pf_base, stats_base = scored({}, strategy=baseline)
pf, stats = scored({})
COLS = ["cagr", "excess_cagr", "alpha_cagr", "beta", "volatility", "sharpe", "max_drawdown",
        "n_trades", "trades_per_year", "pct_profitable", "years"]
display(pd.DataFrame({"defaults": stats, "baseline": stats_base}).loc[COLS].round(4))
plots.plot_equity(bt.equity_curve(pf, init_cash=INIT_CASH),
                  title=f"{strat.name} — equity, config defaults (warm from {SCORE_FROM})").show()
overlay(pf, strat.name, f"{d['universe']} @ {panel.bar} · train — "
        "cumulative return vs equal-weight buy-and-hold").show()

## 4 · Inspect a symbol — trades, levels and the session VWAP

The price panel draws the nearest **support** and **resistance** the strategy read at every bar (the frozen level table of that session, selected by the live close) and the session VWAP; ▲ = long entry, ▼ = short entry, ✕ = exit (level stop, target, time stop or the session-close flat). The volume panel shows relative volume; marked bars are sessions the name held a watchlist seat. A name showing **no trade on a day it clearly bounced** usually lost its seat to a more active name — that is the design, not a bug.

In [ ]:
def inspect_symbol(sym, pf, params):
    """Deep-dive one symbol of a portfolio: annotated chart + its trade table."""
    all_trades = bt.trade_records(pf)
    merged = {**strat.params, **params, "bar": panel.bar, "tz": panel.tz,
              "high": panel.high[sym], "low": panel.low[sym]}
    overlays = strat.indicators(panel.close[sym], panel.volume[sym], **merged)
    plots.plot_symbol(panel.close[sym], panel.volume[sym], all_trades, symbol=sym,
                      overlays=overlays, equity=pf.value(group_by=False)[sym]).show()
    mine = all_trades[all_trades["symbol"] == sym]
    return mine.sort_values("entry_ts").reset_index(drop=True).round(3)

trades = bt.trade_records(pf)
busiest = trades["symbol"].value_counts().index[0] if len(trades) else panel.symbols[0]
inspect_symbol(busiest, pf, strat.params)

## 5 · Ablation sweeps — the gates, and the timeframes

Two small grids rather than the full product. The **gate grid** spans the thesis and its controls one variable at a time: `polarity` × `min_touches` (1 = any swing is a level) × `rank_by` (rvol = activity selection, touches = the no-activity control) × `vol_mult` (0 = no volume floor; with `rank_by: rvol` the floor only binds when fewer than `max_candidates` names clear it). The **timeframe grid** spans `signal_tf` × `trend_tf` (`trend_tf: 0` = no trend gate; otherwise the shipped `trend_gate`) × `direction`. Read the heatmaps for a *region*, not a cell: an edge carried by one combination is noise, and `pct_profitable` (share of names that made money) is the breadth check.

`chunk_size` is lowered on purpose: `bt.sweep` column-stacks combos into one vectorbt call, and 12 × 200 symbols of 1-minute bars exhausts memory. Sweep rows carry no `alpha_cagr` (no per-combo equity curve is kept) and score the whole panel cold, so the maps show `excess_cagr`; the *ordering* is what matters here.

In [ ]:
from itertools import product

def combos(keys, **fixed):
    """The cartesian product of the config grid over `keys`, plus fixed overrides."""
    return [dict(zip(keys, vals, strict=True), **fixed)
            for vals in product(*(strat.sweep[k] for k in keys))]

gate_grid = [g for g in combos(("polarity", "min_touches", "rank_by", "vol_mult"))
             if strat.valid(g)]
tf_grid = [g for g in combos(("signal_tf", "trend_tf", "direction")) if strat.valid(g)]
print(f"{len(gate_grid)} gate combos, {len(tf_grid)} timeframe combos")
sweep_gates = bt.sweep(strat, panel, gate_grid, settings=settings, chunk_size=2)
sweep_tf = bt.sweep(strat, panel, tf_grid, settings=settings, chunk_size=2)
display(sweep_gates.sort_values("excess_cagr", ascending=False).head(8).round(3))
sweep_tf.sort_values("excess_cagr", ascending=False).head(8).round(3)

In [ ]:
plots.plot_heatmap_grid(sweep_gates, pairs=[("polarity", "min_touches"), ("rank_by", "vol_mult")],
                        metrics=("excess_cagr", "total_return", "pct_profitable")).show()
plots.plot_heatmap_grid(sweep_tf, pairs=[("signal_tf", "trend_tf"), ("direction", "trend_tf")],
                        metrics=("excess_cagr", "total_return", "pct_profitable")).show()

## 5b · Event study — measure the trigger *before* trusting the backtest

A backtest is a signal wrapped in stops, a wallet, costs and a benchmark; when it is negative none of those layers says which one is at fault. So measure the naked event first: the bars this config would fill on (the raw trigger at bar *t* fills at *t+1*), the forward return **open-to-open** from that fill, minus the cross-sectional mean of every symbol's return over the identical bars — the market removed — signed as positioned (long as-is, shorts flipped). Positive = the event predicts the position.

**The pre-registered rule** (written before the study ran): the polarity whose **uncapped** row is positive with **t > 2 at both 60 and 120 bars** becomes the shipped `value` of `polarity`. If neither is, §6 is not run; the finding goes into IDEAS.md. Also read:

* **Capped vs uncapped** — the capped row is what trades (≤ `max_candidates` seats); the uncapped row is the signal's whole population and the trustworthy t-stat. They should agree in sign; if the cap flips it, the watchlist ranking is doing the work.
* **The controls** — no activity selection (ranked by touches, no volume floor: with `rank_by: rvol` the ranking *is* the selection, so `vol_mult: 0` alone changes nothing), any-swing levels: each must score *worse* than the thesis, or the ingredient it removes was not earning its place.
* **Touches, relative volume, trend state** — the literature says the bounce probability rises with prior touches and that these signals need elevated activity; the splits say whether this tape agrees.
* **Fill delay** — an effect that only exists at delay 0 is bid-ask bounce. Real effects survive a few bars late.

An edge of a few basis points still has to cross a round trip that costs 6 bps at default costs and ~26 bps at the diagnostics notebook's pass bar; this section is in-sample evidence about a *signal*, never about a strategy.

In [ ]:
import numpy as np

HORIZONS = (15, 30, 60, 120, 180)      # bars held, open-to-open from the fill bar
UNCAPPED = len(panel.symbols)           # a watchlist wide enough to seat everyone


def event_table(params, *, delay=0, split=None):
    """Market-demeaned forward return of the raw triggers one config would fill on.

    `split(parts) -> (bar x symbol) label frame` groups the events (e.g. by touch count).
    """
    parts = compose(panel, **{**strat.params, **params})
    long_, short = parts["long_raw"].to_numpy(), parts["short_raw"].to_numpy()
    labels = None if split is None else split(parts).to_numpy()
    rows = []
    for bars in HORIZONS:
        fill = panel.open_.shift(-(1 + delay))                      # the fill after the signal
        fwd = panel.open_.shift(-(1 + delay + bars)) / fill - 1.0
        excess = fwd.sub(fwd.mean(axis=1), axis=0).to_numpy()       # the market, removed
        signed = np.concatenate([excess[long_], -excess[short]])
        lab = None if labels is None else np.concatenate([labels[long_], labels[short]])
        ok = np.isfinite(signed)
        groups = {"all": ok} if lab is None else {g: ok & (lab == g) for g in pd.unique(lab[ok])}
        for g, m in groups.items():
            v = signed[m]
            rows.append({"bars": bars, "group": g, "events": len(v),
                         "excess_bps": 1e4 * v.mean() if len(v) else np.nan,
                         "t": v.mean() / v.std() * np.sqrt(len(v)) if len(v) > 2 else np.nan})
    out = pd.DataFrame(rows)
    if split is None:
        return out.set_index("bars")[["events", "excess_bps", "t"]]
    return out.pivot(index="bars", columns="group", values=["events", "excess_bps", "t"])


ALL = {"max_candidates": UNCAPPED}
CONFIGS = {
    "bounce, capped":            {"polarity": "bounce"},
    "bounce, uncapped":          {"polarity": "bounce", **ALL},
    "break, uncapped":           {"polarity": "break", **ALL},
    "bounce · no activity selection": {"polarity": "bounce", "rank_by": "touches", "vol_mult": 0.0,
                                       **ALL},
    "bounce · any-swing levels":       {"polarity": "bounce", "min_touches": 1, **ALL},
    "break · no activity selection":   {"polarity": "break", "rank_by": "touches", "vol_mult": 0.0,
                                       **ALL},
}
events = pd.concat({name: event_table(p) for name, p in CONFIGS.items()}, names=["config"])
print("Market-excess return of the raw trigger, signed as the strategy is positioned (bps).")
display(events.round(2))

delayed = pd.concat({f"fill +{k} bars": event_table({"max_candidates": UNCAPPED}, delay=k)
                     for k in (0, 2, 5)}, names=["control"])
print("\nFill-delay control (bounce, uncapped): an effect that survives a late fill is "
      "information.")
display(delayed.round(2))


def passes(name):
    """The pre-registered rule: t > 2 at both 60 and 120 bars, uncapped."""
    t = events.loc[name, "t"]
    return bool(t.loc[60] > 2 and t.loc[120] > 2)


verdict = {pol: passes(f"{pol}, uncapped") for pol in ("bounce", "break")}
winners = [pol for pol, ok in verdict.items() if ok]
print(f"\npre-registered rule (t > 2 at 60 AND 120 bars, uncapped): {verdict}")
print("→ ship polarity:", winners[0] if len(winners) == 1 else
      ("BOTH pass — keep it searched" if winners else
       "NEITHER passes — write it up in IDEAS.md, do not run §6"))

In [ ]:
# Where does the (bounce) edge live? By the traded level's touch count, by relative volume at
# the scan, and by the state of a 15-minute EMA trend — the splits the literature predicts.
UNC = {"polarity": "bounce", "max_candidates": UNCAPPED}
trend15 = compose(panel, **{**strat.params, **UNC, "trend_tf": 15, "trend_len": 20,
                            "trend_gate": "with"})["trend"]


def by_touches(parts):
    n = parts["sup_touches"].where(parts["long_raw"], parts["res_touches"])
    return pd.DataFrame(np.select([n <= 2, n <= 4], ["2 touches", "3-4 touches"], "5+ touches"),
                        index=n.index, columns=n.columns)


def by_rvol(parts):
    r = parts["rvol"]
    return pd.DataFrame(np.select([r < 1.0, r < 2.0], ["rvol < 1", "rvol 1-2"], "rvol ≥ 2"),
                        index=r.index, columns=r.columns)


def by_trend(parts):
    side = np.where(parts["long_raw"].to_numpy(), 1, -1)
    aligned = np.sign(trend15.to_numpy()) * side
    return pd.DataFrame(np.select([aligned > 0, aligned < 0], ["with trend", "against trend"],
                                  "no trend"), index=trend15.index, columns=trend15.columns)


print("bounce, uncapped — by the traded level's touch count")
display(event_table(UNC, split=by_touches).round(2))
print("\nbounce, uncapped, no volume floor — by relative volume at the scan")
display(event_table({**UNC, "vol_mult": 0.0}, split=by_rvol).round(2))
print("\nbounce, uncapped — by the 15-minute EMA(20) trend of completed candles (gate off)")
event_table(UNC, split=by_trend).round(2)

## 6 · Optimize (resumable, runs outside the notebook)

`jobs.run_optimize` hands the work to a `tsim optimize` subprocess. The study scores **`alpha_cagr` (maximized) against `volatility` (minimized)** — the return left after the beta the book actually ran, and how bumpy its ride is — every trial **warm from `optuna.score_from`**, with `min_trades_per_year` pruning the lucky-few-trades artifacts. The pick only ever chooses a front trial with positive alpha. Every finished trial is checkpointed to `studies/{study}.log`; **re-running this cell resumes**. One worker only: seeded parallel workers draw duplicate trials (a `momentum_swing` finding).

Do not run this until §5b has named a polarity — and set `config.yaml`'s `polarity.value` to it first (the search stays on both, so the study can still disagree).

In [ ]:
N_TRIALS = 1000   # raise for a deeper search; re-runs accumulate into the same study
jobs.run_optimize("level_flow", n_trials=N_TRIALS)

## 7 · Read the study: Pareto front, the picked trial, the candidates file

Gray dots are all completed trials; the staircase is the Pareto front; the star is `opt.pick_from_front`'s choice under the repo-wide balanced weights, among front rows above the **picking floor** (`optuna.pick_min_trades_per_year`, the same floor the walk-forward re-picks with) and above the alpha hurdle. Read the prune rate first: a front made of a few trials that survived the trade floor is not a front.

Every distinct admissible front row is saved to `studies/<study>.candidates.yaml` (git-ignored), which is what the diagnostics notebook reads.

In [ ]:
import datetime as _dt
import yaml

study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)
SPACE = strat.optuna["space"]
SPACE_INT = {k for k, spec in SPACE.items() if spec["type"] == "int"}
objs = list(opt.objectives_for(strat))


def as_params(row, keys=None) -> dict:
    """A front row's searched parameters, cast back to the types signals() expects."""
    out = {}
    for k in (keys or SPACE):
        v = row[k]
        if isinstance(v, str):
            out[k] = v
        elif isinstance(v, (bool, np.bool_)):
            out[k] = bool(v)
        elif SPACE[k]["type"] == "categorical":
            out[k] = type(SPACE[k]["choices"][0])(v)
        elif k in SPACE_INT:
            out[k] = int(round(float(v)))
        else:
            out[k] = round(float(v), 10)
    return out


states = pd.Series([t.state.name for t in study.trials]).value_counts()
print(f"{len(study.trials)} trials — " + ", ".join(f"{n} {s.lower()}" for s, n in states.items()))
eligible = front[front["trades_per_year"] >= FLOOR]
if eligible.empty:
    print(f"!! no Pareto trial reaches {FLOOR} trades/yr — showing the raw front; "
          "treat any pick as inadmissible")
    eligible = front
best_row = opt.pick_from_front(eligible)
best = next(t for t in study.best_trials if t.number == int(best_row["trial"]))
print(f"front: {len(front)} trials, {len(eligible)} admissible (>= {FLOOR}/yr), "
      f"{int(best_row['front_distinct'])} distinct outcomes; picked trial {best.number}"
      + (f" — TIED with {int(best_row['n_tied']) - 1} other(s)" if best_row["n_tied"] > 1 else "")
      + ("" if best_row["beats_benchmark"] else " — NOTHING on the front has positive alpha"))
for k in ("polarity", "direction", "signal_tf", "trend_tf", "trend_gate"):
    if k in front.columns:
        print(f"front by {k}:", front[k].value_counts().to_dict(),
              f"— picked: {best.params.get(k)}")
plots.plot_pareto(front, everything, picked=best.number).show()

distinct = opt.distinct_front(eligible)
METRICS = ["alpha_cagr", "excess_cagr", "beta", "volatility", "sharpe", "max_drawdown",
           "n_trades", "trades_per_year"]
payload = {"study": study.study_name, "saved": _dt.datetime.now().isoformat(timespec="seconds"),
           "pick": int(best.number), "pick_min_trades_per_year": FLOOR,
           "objectives": objs, "score_from": SCORE_FROM,
           "front": [{"trial": int(r["trial"]), "params": as_params(r),
                      **{m: float(r[m]) for m in METRICS if m in r}}
                     for _, r in distinct.iterrows()]}
cand_path = opt.study_path(strat, study.study_name).with_suffix(".candidates.yaml")
cand_path.write_text(yaml.safe_dump(payload, sort_keys=False))
print(f"\nsaved {len(payload['front'])} candidates → {cand_path.name}")
eligible.head(8).round(3)

## 8 · Boundary check — did the optimum hit the search-space edge?

A picked trial on the edge of a searched range hints that the true optimum lies outside it. If something is flagged: widen that bound in `config.yaml`, bump `optuna.study` (a journal that mixes two spaces cannot be read), re-run §6, re-read §7–§8 until the front sits inside the box.

In [ ]:
report = opt.boundary_report(best.params, strat.optuna["space"], front=front)
flagged = report[report["action"] != "ok"]
print("all clear — the searched ranges bracket the optimum" if flagged.empty
      else f"{len(flagged)} parameter(s) pressing against the search space:")
report

## 9 · Re-run the full backtest with the picked parameters — against the baseline

In [ ]:
picked = as_params(best_row)
print("picked params:", picked)
pf_best, stats_best = scored(picked)
display(pd.DataFrame({"defaults (§3)": stats, "picked": stats_best, "baseline": stats_base})
        .loc[COLS].round(4))
plots.plot_equity(bt.equity_curve(pf_best, init_cash=INIT_CASH),
                  title=f"{strat.name} — equity, picked parameters (warm from {SCORE_FROM})").show()
overlay(pf_best, f"{strat.name} (picked)",
        f"{d['universe']} @ {panel.bar} · train — picked vs equal-weight buy-and-hold").show()

## 10 · Verification — per-symbol deep dive

Portfolio numbers can hide an edge carried by a single name. First the per-symbol breakdown of the picked backtest, then set `SYMBOL` to any name that looks interesting (the big winner, the one loser, the one with suspiciously few trades) for the annotated chart and trade table of §4.

In [ ]:
bt.symbol_stats(pf_best).round(4)

In [ ]:
SYMBOL = None   # ← set to any symbol from the table above; None = most-traded
trades_best = bt.trade_records(pf_best)
sym = SYMBOL or (trades_best["symbol"].value_counts().index[0] if len(trades_best)
                 else panel.symbols[0])
inspect_symbol(sym, pf_best, picked)

## Read the result honestly

* §3–§10 are **in-sample**: the optimizer saw all the data they are judged on. The out-of-sample evidence lives in [`level_flow_diagnostics.ipynb`](./level_flow_diagnostics.ipynb): the rolling walk-forward (refit vs frozen vs defaults, stitched OOS path), the untouched May–June 2026 holdout, the 10/3 bps pass bar, pessimistic fills, the verdict.
* **§5b outranks §9.** A backtest can be dragged positive by a lucky stop placement or one runaway name; the event study measures the thing the strategy is betting on, over thousands of events, with the market removed. If they disagree, believe the event study.
* `excess_cagr` will be negative and that is expected: a book flat most of the day cannot out-earn a fully invested +47%/yr basket. The question this strategy asks is whether its **selection** has value after the beta it runs (`alpha_cagr`) — and whether that survives costs, unseen windows and re-tuning.
* Prefer combos where **`pct_profitable` is high**: on a market-wide day the watchlist fills with one correlated bet, so an edge carried by one symbol is usually noise.
* The pre-registered rules in the README decide what ships. Anything less ships as a finding, not a config.